In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Copy datasets from the read-only input mount into the WRITABLE working dir.
# (The synthetic loaders write points3d.ply into the scene dir, so the source MUST
#  be writable — /kaggle/input is read-only.)
WORK=/kaggle/working/spec-fastgs/spec-fastgs/datasets
mkdir -p "$WORK"

# Mip-NeRF 360 — auto-locate under /kaggle/input by folder name, regardless of the
# exact dataset slug/nesting Kaggle mounts it under (avoids hardcoding a path that
# only matches one specific attached-dataset naming convention).
MIPNERF_SRC=$(find /kaggle/input -maxdepth 5 -type d -name "mipnerf360" | head -1)
if [ -n "$MIPNERF_SRC" ]; then
    echo "📥 copying $MIPNERF_SRC -> $WORK/mipnerf360"
    cp -r "$MIPNERF_SRC" "$WORK/"
else
    echo "⚠️  mipnerf360 NOT found under /kaggle/input — check the attached dataset"
fi

# Synthetic suites — auto-locate under /kaggle/input regardless of the dataset slug.
for d in Anisotropic-Synthetic-Dataset Synthetic_NSVF; do
    SRC=$(find /kaggle/input -maxdepth 4 -type d -name "$d" | head -1)
    if [ -n "$SRC" ]; then
        echo "📥 copying $SRC -> $WORK/"
        cp -r "$SRC" "$WORK/"
    else
        echo "⚠️  $d NOT found under /kaggle/input"
    fi
done
echo "📂 datasets now in working:"; ls "$WORK"

In [ ]:
%%bash
# ============================================================
# R8 PREREQUISITE — sweep the counter dataset with the Tan-Ikeuchi specular-prior
# detector (v3.2). Pure CPU/numpy/scipy/PIL — NO GPU, no trained model, no pip installs
# needed (Kaggle's stock python ships the full scipy stack). Unlike the monocular normal-
# prior sweep, this does NOT need to run before the conda teardown for any technical
# reason — it's placed here for convenience (datasets are already copied to the writable
# working dir by the cell above). Takes ~0.6s/image (~150s for counter's 240 images).
#
# Writes <source>/tanikeuchi_priors/<image_name>.png (boolean mask, uint8 PNG, 0/255) +
# a red-overlay preview for the first 5 images, so you can eyeball it immediately.
# ============================================================
set -e
cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Align the dataset layout NOW (same guard as the training cells; idempotent)
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi
ls -d ./datasets/mipnerf360/counter/images >/dev/null && echo "✅ dataset ready"

# defensive install — Kaggle's stock python almost certainly already has these
pip install -q scipy Pillow numpy

python tools/gen_tanikeuchi_priors.py \
    -s ./datasets/mipnerf360/counter \
    -i images \
    --preview 5

echo "🧭 tanikeuchi priors written:"
ls ./datasets/mipnerf360/counter/tanikeuchi_priors | head -5
echo "... total:" $(ls ./datasets/mipnerf360/counter/tanikeuchi_priors/*.png | grep -v preview | wc -l) "png priors"


In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm imageio

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
%%bash
# ============================================================
# RUN R8 (v3.2) — PRECOMPUTED TAN-IKEUCHI SPECULAR LOCATOR on counter (Mip-NeRF)
# Requires the "R8 PREREQUISITE" cell above to have written
#   ./datasets/mipnerf360/counter/tanikeuchi_priors/*.png
# This is the real test: does swapping R7's model-residual locator for a precomputed,
# model-independent one finally move NCC/sigma/energyRatio on a REAL scene (R7 alone
# was neutral here). Outputs -> ./output/counter_r8
#
# NOTE (2026-07-05): the Tan-Ikeuchi locator was found to lack an explicit
# desaturation gate that the prior (v3.1) Shafer locator had — fixed by adding
# S < sat_thresh. Full 240-image sweep after the fix: mean=3.08%/max=6.84% flagged
# (down from mean=7.89%/max=17.96% before the fix), now much closer to Shafer's own
# mean=1.44%/max=4.22%. Also fixed: a corroboration check in the densification vote
# so appearance-flagged pixels only get excluded from geometric credit once the
# specular branch is measurably explaining them (protects against both early-training
# starvation and misclassified diffuse objects). Still watch Gaussian count / PSNR-SSIM
# for any residual dilution effect versus the prior Shafer-based run.
# ============================================================
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Dataset layout guard (idempotent — already aligned by the prereq cell)
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

# 2. Kick off R8. Watch the banner for 'code: v3.2-...' and
#    'spec_densify=True (w=0.5, locator=tanikeuchi)'. The stale-code guard and the
#    tanikeuchi-priors safety check will abort with a clear message if either is missing.
bash run_spec-fastgs_big_r8.sh


In [ ]:
import shutil, os
# Archive ONLY the R8 output (one run per session — keeps the zip small and avoids
# filling the disk, which is what killed the multi-run Version 28).
src = '/kaggle/working/spec-fastgs/spec-fastgs/output/counter_r8'
out = '/kaggle/working/spec_fastgs_output_r8'
if os.path.isdir(src):
    shutil.make_archive(out, 'zip', src)
    print('archived:', out + '.zip', round(os.path.getsize(out + '.zip')/1e6, 1), 'MB')
else:
    print('no counter_r8 output found at', src)
